In [3]:
!pip install opencv-python pillow torch torchvision numpy

You should consider upgrading via the 'c:\users\i n t e l\appdata\local\programs\python\python39\python.exe -m pip install --upgrade pip' command.


In [4]:
import cv2
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from collections import deque, Counter

from torchvision import models, transforms

ValueError: source code string cannot contain null bytes

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

MODEL_PATH = r"C:\Users\I N T E L\Desktop\FER_Evaluation\FER_VGG19\best.pth"
NUM_CLASSES = 7

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

Using device: cpu


In [ ]:
def get_vgg19_model():
    model = models.vgg19(weights=None)
    model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    return model.to(DEVICE)

model = get_vgg19_model()
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print("VGG19 model loaded successfully.")

VGG19 model loaded successfully.


In [ ]:
live_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485]*3, [0.229]*3)
])

NameError: name 'transforms' is not defined

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

print("Face detector loaded.")

Face detector loaded.


In [ ]:
def preprocess_face(face_bgr):
    # Convert BGR -> grayscale
    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)

    # Convert grayscale -> PIL image
    pil_img = Image.fromarray(gray)

    # Convert grayscale PIL -> RGB PIL to match training input format
    pil_img = pil_img.convert("RGB")

    # Apply transforms
    tensor = live_transform(pil_img).unsqueeze(0).to(DEVICE)

    return tensor

In [ ]:
def predict_emotion(face_bgr):
    x = preprocess_face(face_bgr)

    with torch.no_grad():
        outputs = model(x)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_label = emotion_labels[pred_idx]
    confidence = float(probs[pred_idx])

    return pred_label, confidence, probs

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

# store last few predictions for smoothing
emotion_buffer = deque(maxlen=10)
confidence_buffer = deque(maxlen=10)

print("Press Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # mirror view for natural interaction
    frame = cv2.flip(frame, 1)

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray_frame,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(80, 80)
    )

    if len(faces) > 0:
        # choose the largest detected face
        faces = sorted(faces, key=lambda b: b[2] * b[3], reverse=True)
        x, y, w, h = faces[0]

        face_crop = frame[y:y+h, x:x+w]

        try:
            emotion, conf, probs = predict_emotion(face_crop)

            emotion_buffer.append(emotion)
            confidence_buffer.append(conf)

            # majority vote smoothing
            smoothed_emotion = Counter(emotion_buffer).most_common(1)[0][0]
            smoothed_conf = float(np.mean(confidence_buffer))

            # draw face box
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

            # draw label
            text = f"{smoothed_emotion} ({smoothed_conf:.2f})"
            cv2.putText(
                frame,
                text,
                (x, y-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

        except Exception as e:
            cv2.putText(
                frame,
                "Prediction error",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

    else:
        cv2.putText(
            frame,
            "No face detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    cv2.imshow("Live FER - VGG19", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Press Q to quit.
